# Teks — Topic Modeling dengan BERTopic (embedding kalimat + clustering)
**Kapan dipakai:** ingin topik yang lebih "bermakna" daripada LDA/NMF (`text/topik`),
terutama untuk teks pendek. Jumlah topik tidak wajib ditentukan di awal.

**Cara kerja:**
1. Embedding kalimat (MiniLM lokal) → tiap dokumen jadi vektor angka.
2. UMAP mengecilkan dimensi, HDBSCAN mengelompokkan.
3. Kata khas tiap kelompok dicari dengan c-TF-IDF.

Dokumen yang tidak masuk kelompok mana pun = topik -1.

**Data demo:** news4 (4 topik). Label HANYA dipakai untuk membandingkan (NMI/ARI).

In [1]:
import pandas as pd
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from umap import UMAP

## Konfigurasi

In [2]:
DATA_FOLDER = "../../data_latihan/text/classification_en"   # GANTI saat lomba
DATA_CSV = f"{DATA_FOLDER}/news4_train.csv"
EMBEDDING_MODEL = "../../pretrained/minilm_en"
# teks Indonesia: EMBEDDING_MODEL = "../../pretrained/minilm"

TEXT_COL = "text"
LABEL_COL = "label"       # hanya untuk evaluasi, hapus bila soal tidak punya label
MIN_TOPIC_SIZE = 50       # kelompok lebih kecil dari ini digabung / jadi -1
N_TOPICS = 5              # target topik + 1 (topik -1 ikut dihitung). None = otomatis
SEED = 42

## Baca data

In [3]:
data = pd.read_csv(DATA_CSV)
data[TEXT_COL] = data[TEXT_COL].fillna("")
texts = list(data[TEXT_COL])
print(data.shape)
print(data[LABEL_COL].value_counts())

(2140, 3)
label
sci.med               573
comp.graphics         565
rec.autos             552
talk.politics.misc    450
Name: count, dtype: int64


## Embedding kalimat
Dihitung sekali; saat mencoba pengaturan BERTopic lain, sel ini tidak perlu dijalankan ulang.

In [4]:
encoder = SentenceTransformer(EMBEDDING_MODEL)
embeddings = encoder.encode(texts, batch_size=64, show_progress_bar=False)
print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(2140, 384)


## BERTopic
`random_state` di UMAP supaya hasil bisa diulang. Stopword dibuang hanya saat memilih kata
topik.

In [5]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine",
                  random_state=SEED)
hdbscan_model = HDBSCAN(min_cluster_size=MIN_TOPIC_SIZE, metric="euclidean",
                        prediction_data=True)
word_counter = CountVectorizer(stop_words="english", min_df=2)
# teks Indonesia: word_counter = CountVectorizer(stop_words=daftar_stopword_id, min_df=2)

model = BERTopic(
    embedding_model=encoder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=word_counter,
    nr_topics=N_TOPICS,
)
topics, probabilities = model.fit_transform(texts, embeddings)
topic_info = model.get_topic_info()
topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,112,-1_edu_keyboard_graphics_pub,"[edu, keyboard, graphics, pub, drugs, mail, ra...",[Archive-name: typing-injury-faq/general Versi...
1,0,518,0_car_cars_like_just,"[car, cars, like, just, engine, good, don, spe...","[While I don't read normally read this group, ..."
2,1,516,1_image_jpeg_graphics_file,"[image, jpeg, graphics, file, data, images, so...",[I have posted disp135.zip to alt.binaries.pic...
3,2,515,2_medical_patients_health_msg,"[medical, patients, health, msg, people, don, ...",[------------- cut here ----------------- Univ...
4,3,479,3_mr_president_people_think,"[mr, president, people, think, don, government...",[THE WHITE HOUSE Office o...


## Kata kunci tiap topik

In [6]:
for topic_number in topic_info["Topic"]:
    if topic_number == -1:
        continue
    words = []
    for word, weight in model.get_topic(topic_number)[:10]:
        words.append(word)
    print("topik", topic_number, ":", ", ".join(words))

topik 0 : car, cars, like, just, engine, good, don, speed, new, oil
topik 1 : image, jpeg, graphics, file, data, images, software, color, files, use
topik 2 : medical, patients, health, msg, people, don, food, doctor, disease, cancer
topik 3 : mr, president, people, think, don, government, know, going, just, com


## Evaluasi dengan label (HANYA DEMO; saat lomba biasanya tidak ada label)
Pembanding: LDA di `text/topik` (NMI 0.41, ARI 0.39).

In [7]:
data["topik"] = topics
print("dokumen tanpa topik (-1):", round((data["topik"] == -1).mean(), 3))
print("NMI:", round(normalized_mutual_info_score(data[LABEL_COL], data["topik"]), 3))
print("ARI:", round(adjusted_rand_score(data[LABEL_COL], data["topik"]), 3))
print(pd.crosstab(data[LABEL_COL], data["topik"]))

dokumen tanpa topik (-1): 0.052
NMI: 0.714
ARI: 0.764
topik               -1    0    1    2    3
label                                     
comp.graphics       26    3  505    0   31
rec.autos            6  501    3    5   37
sci.med             51    6    8  486   22
talk.politics.misc  29    8    0   24  389


## Simpan topik tiap dokumen + info topik

In [8]:
result = pd.DataFrame()
result["text"] = data[TEXT_COL]
result["topik"] = data["topik"]
result.to_csv("topik_dokumen.csv", index=False)
topic_info.to_csv("info_topik.csv", index=False)
result.head()

,text,topik
0,What kind of polygons? Shaded? Texturemapped...,1
1,"Graeme> \tYes, that's known as ""Bresenhams Run...",1
2,Are you using the traditional radiosity method...,1
3,I need to bring on my VW Corrado for body work...,0
4,If she is having problems with fresh vegetable...,2
